# MCTS hot-path sampling benchmark

Benchmarks every sampling strategy used in `pop_untried`, `weighted_sample_row`, and row materialisation. Run all cells top to bottom.

## 0 — imports and synthetic data

In [1]:
import timeit
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
from numpy.random import default_rng

rng = default_rng(42)
np.random.seed(42)

SMALL  = 50
MEDIUM = 500
LARGE  = 5_000
N      = 10_000   # timeit repetitions per benchmark
DRAIN_STEPS = 50  # pops per drain — mimics one node's full expansion
DRAIN_N     = 200 # drain benchmark repetitions
ROW_COLS = ("unique_id", "block", "end_tag", "frequency")

def make_pool(n, seed=0):
    """Synthetic Arrow table matching your fragment pool schema."""
    r = default_rng(seed)
    freqs = r.zipf(1.8, n).astype(np.float32)   # heavy-tailed like MOSES
    return pa.table({
        "unique_id": [f"id_{i}" for i in range(n)],
        "block":     [f"C{i}"   for i in range(n)],
        "end_tag":   ["1"] * n,
        "frequency": freqs.tolist(),
    })

def make_weights(pool, temperature=1.0):
    freqs = np.array(pool.column("frequency").to_pylist(), dtype=np.float32)
    log_f = np.log1p(freqs) / temperature
    log_f -= log_f.max()
    w = np.exp(log_f)
    return w / w.sum()

pool_s = make_pool(SMALL)
pool_m = make_pool(MEDIUM)
pool_l = make_pool(LARGE)
w_s = make_weights(pool_s)
w_m = make_weights(pool_m)
w_l = make_weights(pool_l)

def bench(fn, n=N):
    return timeit.timeit(fn, number=n) / n * 1e6   # returns µs/call

print(f"pools ready: small={SMALL}  medium={MEDIUM}  large={LARGE}")

pools ready: small=50  medium=500  large=5000


## 1 — index selection strategies

Single weighted draw from a fixed distribution — called inside `pop_untried` and `weighted_sample_row` on every iteration.

### A — `np.random.choice(p=)` *(current)*

In [2]:
def sample_choice(w):
    return int(np.random.choice(len(w), p=w))

for label, w in [("small", w_s), ("medium", w_m), ("large", w_l)]:
    t = bench(lambda w=w: sample_choice(w))
    print(f"np.random.choice   [{label:6s} n={len(w):5d}]  {t:8.2f} µs/call")

np.random.choice   [small  n=   50]      6.74 µs/call
np.random.choice   [medium n=  500]      6.11 µs/call
np.random.choice   [large  n= 5000]     26.78 µs/call


### B — `default_rng().choice(p=)`

In [3]:
_rng = default_rng(42)

def sample_rng_choice(w):
    return int(_rng.choice(len(w), p=w))

for label, w in [("small", w_s), ("medium", w_m), ("large", w_l)]:
    t = bench(lambda w=w: sample_rng_choice(w))
    print(f"default_rng.choice [{label:6s} n={len(w):5d}]  {t:8.2f} µs/call")

default_rng.choice [small  n=   50]      5.82 µs/call
default_rng.choice [medium n=  500]      7.35 µs/call
default_rng.choice [large  n= 5000]     26.45 µs/call


### C — `searchsorted` / inverse-CDF  
Build a CDF once per node, draw a uniform, binary search. O(log n) per draw.

In [4]:
cdf_s = np.cumsum(w_s)
cdf_m = np.cumsum(w_m)
cdf_l = np.cumsum(w_l)

def sample_searchsorted(cdf):
    return int(np.searchsorted(cdf, np.random.random()))

for label, cdf in [("small", cdf_s), ("medium", cdf_m), ("large", cdf_l)]:
    t = bench(lambda cdf=cdf: sample_searchsorted(cdf))
    print(f"searchsorted       [{label:6s} n={len(cdf):5d}]  {t:8.2f} µs/call")

searchsorted       [small  n=   50]      1.57 µs/call
searchsorted       [medium n=  500]      1.61 µs/call
searchsorted       [large  n= 5000]      1.71 µs/call


### D — Vose alias method  
O(n) one-time setup, O(1) draw. Best when many draws are made from the same distribution.

In [5]:
class AliasTable:
    def __init__(self, weights):
        n   = len(weights)
        w   = np.array(weights, dtype=np.float64)
        w   = w / w.sum() * n
        self.prob  = np.zeros(n)
        self.alias = np.zeros(n, dtype=np.int64)
        small, large = [], []
        for i, p in enumerate(w):
            (small if p < 1.0 else large).append(i)
        while small and large:
            s, l = small.pop(), large.pop()
            self.prob[s]  = w[s]
            self.alias[s] = l
            w[l] = (w[l] + w[s]) - 1.0
            (small if w[l] < 1.0 else large).append(l)
        for i in large + small:
            self.prob[i] = 1.0

    def sample(self):
        i = np.random.randint(len(self.prob))
        return i if np.random.random() < self.prob[i] else int(self.alias[i])

alias_s = AliasTable(w_s)
alias_m = AliasTable(w_m)
alias_l = AliasTable(w_l)

for label, a in [("small", alias_s), ("medium", alias_m), ("large", alias_l)]:
    t = bench(lambda a=a: a.sample())
    print(f"alias O(1)         [{label:6s} n={len(a.prob):5d}]  {t:8.2f} µs/call")

alias O(1)         [small  n=   50]      1.56 µs/call
alias O(1)         [medium n=  500]      1.34 µs/call
alias O(1)         [large  n= 5000]      1.20 µs/call


## 2 — pool-draining strategies

`pop_untried` drains the candidate pool by removing one row per call. This runs up to `target_n_blocks` times per node. Tested on medium pool, 50 pops.

### E — Arrow slice + concat  *(current)*

In [6]:
def pop_arrow_slice(pool, idx):
    if pool.num_rows <= 1:
        return pool.slice(0, 0)
    return pa.concat_tables([pool.slice(0, idx), pool.slice(idx + 1)])

def drain_arrow_slice(pool):
    w   = make_weights(pool)
    cdf = np.cumsum(w)
    for _ in range(min(DRAIN_STEPS, pool.num_rows)):
        idx  = min(int(np.searchsorted(cdf, np.random.random())), pool.num_rows - 1)
        pool = pop_arrow_slice(pool, idx)
        if pool.num_rows == 0:
            break
        w   = make_weights(pool)
        cdf = np.cumsum(w)

t = timeit.timeit(lambda: drain_arrow_slice(make_pool(MEDIUM)), number=DRAIN_N)
print(f"E  Arrow slice+concat   [medium n={MEDIUM}]  {t/DRAIN_N*1000:.3f} ms/drain")

E  Arrow slice+concat   [medium n=500]  5.417 ms/drain


### F — bool mask on numpy array

In [7]:
def drain_bool_mask(pool, weights):
    mask = np.ones(pool.num_rows, dtype=bool)
    w    = weights.copy()
    for _ in range(min(DRAIN_STEPS, pool.num_rows)):
        active = w * mask
        total  = active.sum()
        if total == 0:
            break
        active /= total
        idx     = min(int(np.searchsorted(np.cumsum(active), np.random.random())),
                      pool.num_rows - 1)
        mask[idx] = False
        w[idx]    = 0.0

t = timeit.timeit(lambda: drain_bool_mask(pool_m, w_m), number=DRAIN_N)
print(f"F  bool mask numpy      [medium n={MEDIUM}]  {t/DRAIN_N*1000:.3f} ms/drain")

F  bool mask numpy      [medium n=500]  0.322 ms/drain


### G — zero-weight numpy array (no mask needed)

In [8]:
def drain_zero_weight(weights):
    w = weights.copy()
    for _ in range(min(DRAIN_STEPS, len(w))):
        total = w.sum()
        if total == 0:
            break
        w_norm = w / total
        idx    = min(int(np.searchsorted(np.cumsum(w_norm), np.random.random())),
                     len(w) - 1)
        w[idx] = 0.0

t = timeit.timeit(lambda: drain_zero_weight(w_m), number=DRAIN_N)
print(f"G  zero-weight numpy    [medium n={MEDIUM}]  {t/DRAIN_N*1000:.3f} ms/drain")

G  zero-weight numpy    [medium n=500]  0.288 ms/drain


## 3 — row materialisation strategies

Once an index is chosen, a row dict is built and returned to the MCTS phase functions. Called on every expand and every rollout step.

### H — column-by-column `as_py()`  *(current)*

In [9]:
def materialise_aspyy(pool, idx):
    return {col: pool.column(col)[idx].as_py() for col in ROW_COLS}

for label, pool in [("small", pool_s), ("medium", pool_m), ("large", pool_l)]:
    idx = pool.num_rows // 2
    t   = bench(lambda pool=pool, idx=idx: materialise_aspyy(pool, idx))
    print(f"H  as_py per col   [{label:6s} n={pool.num_rows:5d}]  {t:8.2f} µs/call")

H  as_py per col   [small  n=   50]      4.23 µs/call
H  as_py per col   [medium n=  500]      3.89 µs/call
H  as_py per col   [large  n= 5000]      2.64 µs/call


### I — pre-materialised Python lists

In [10]:
class CachedPool:
    """All columns materialised once as lists / numpy arrays at construction."""
    def __init__(self, pool):
        self.unique_id = pool.column("unique_id").to_pylist()
        self.block     = pool.column("block").to_pylist()
        self.end_tag   = pool.column("end_tag").to_pylist()
        self.frequency = np.array(pool.column("frequency").to_pylist(), dtype=np.float32)

    def get_row(self, idx):
        return {
            "unique_id": self.unique_id[idx],
            "block":     self.block[idx],
            "end_tag":   self.end_tag[idx],
            "frequency": float(self.frequency[idx]),
        }

cached_s = CachedPool(pool_s)
cached_m = CachedPool(pool_m)
cached_l = CachedPool(pool_l)

for label, c in [("small", cached_s), ("medium", cached_m), ("large", cached_l)]:
    idx = len(c.unique_id) // 2
    t   = bench(lambda c=c, idx=idx: c.get_row(idx))
    print(f"I  pre-materialised [{label:6s} n={len(c.unique_id):5d}]  {t:8.2f} µs/call")

I  pre-materialised [small  n=   50]      0.20 µs/call
I  pre-materialised [medium n=  500]      0.20 µs/call
I  pre-materialised [large  n= 5000]      0.18 µs/call


## 4 — weight construction cost

Building the softmax weights from frequency data — paid once per node in expand and once per rollout step in simulate.

In [11]:
freq_s = np.array(pool_s.column("frequency").to_pylist(), dtype=np.float32)
freq_m = np.array(pool_m.column("frequency").to_pylist(), dtype=np.float32)
freq_l = np.array(pool_l.column("frequency").to_pylist(), dtype=np.float32)

def make_weights_numpy(freqs, temperature=1.0):
    log_f = np.log1p(freqs) / temperature
    log_f = log_f - log_f.max()
    w = np.exp(log_f)
    return w / w.sum()

logw_s = np.log1p(freq_s); logw_s -= logw_s.max()
logw_m = np.log1p(freq_m); logw_m -= logw_m.max()
logw_l = np.log1p(freq_l); logw_l -= logw_l.max()

def make_cdf_from_logw(logw):
    w = np.exp(logw); w /= w.sum()
    return np.cumsum(w)

for label, pool, fn_arrow, freqs, logw in [
    ("small",  pool_s, pool_s, freq_s, logw_s),
    ("medium", pool_m, pool_m, freq_m, logw_m),
    ("large",  pool_l, pool_l, freq_l, logw_l),
]:
    tj = bench(lambda pool=pool: make_weights(pool))
    tk = bench(lambda freqs=freqs: make_weights_numpy(freqs))
    tl = bench(lambda logw=logw:  make_cdf_from_logw(logw))
    print(f"[{label:6s} n={len(freqs):5d}]  "
          f"J Arrow {tj:7.2f} µs  "
          f"K numpy {tk:7.2f} µs  "
          f"L pre-logw {tl:7.2f} µs")

[small  n=   50]  J Arrow   12.67 µs  K numpy    2.20 µs  L pre-logw    1.62 µs
[medium n=  500]  J Arrow   69.72 µs  K numpy    3.70 µs  L pre-logw    3.03 µs
[large  n= 5000]  J Arrow  700.72 µs  K numpy   18.13 µs  L pre-logw   17.39 µs


## 5 — head-to-head summary table

In [12]:
print(f"{'Strategy':<36} {'small µs':>9} {'medium µs':>10} {'large µs':>9}")
print("-" * 68)

strategies = {
    "A  np.random.choice(p=)  [current]": [
        lambda: sample_choice(w_s),
        lambda: sample_choice(w_m),
        lambda: sample_choice(w_l),
    ],
    "B  default_rng.choice(p=)": [
        lambda: sample_rng_choice(w_s),
        lambda: sample_rng_choice(w_m),
        lambda: sample_rng_choice(w_l),
    ],
    "C  searchsorted (inv-CDF)": [
        lambda: sample_searchsorted(cdf_s),
        lambda: sample_searchsorted(cdf_m),
        lambda: sample_searchsorted(cdf_l),
    ],
    "D  alias O(1) draw": [
        lambda: alias_s.sample(),
        lambda: alias_m.sample(),
        lambda: alias_l.sample(),
    ],
}

for name, fns in strategies.items():
    times = [bench(fn) for fn in fns]
    print(f"{name:<36} {times[0]:>9.2f} {times[1]:>10.2f} {times[2]:>9.2f}")

print("-" * 68)
print("(all times in µs per call)")

Strategy                              small µs  medium µs  large µs
--------------------------------------------------------------------
A  np.random.choice(p=)  [current]        6.47       5.76     26.21
B  default_rng.choice(p=)                 3.74       5.76     25.57
C  searchsorted (inv-CDF)                 0.75       0.77      1.42
D  alias O(1) draw                        0.85       0.91      0.93
--------------------------------------------------------------------
(all times in µs per call)
